# Tone Attunement testing

The purpose of this notebook is to test the custom tone_attunement metric with both a subjectively good and bad answer. The input question was extracted directly from interviews with Red Cross members who have been in direct contact with migrants. The expected and correct outputs were taken from the `goldens` dataset ('correct' term used figuratively, as a number of answers can be technically correct - i.e., respect the HIA database - while not being in-line with the values the Red Cross wants the chatbot to respect), while the incorrect output was manually written.

In [16]:
import os
import pandas as pd
import numpy as np
from rich import print
import ast
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from dotenv import load_dotenv, dotenv_values
from deepeval.models import AzureOpenAIModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

In [17]:
goldens = pd.read_csv("../synthetic_data/goldens.csv")

In [18]:
endpoint = "https://510-ai-research.openai.azure.com/"
model = "gpt-4.1"
deployment = "gpt-4.1-students"

# loading variables from .env file
load_dotenv(r"C:\Users\dari\Desktop\grad project 2026\HIA-eval-framework\.env")
subscription_key = os.getenv("azure_subscription_key")
api_version = "2024-12-01-preview"

# custom model
custom_model = AzureOpenAIModel(
  model=deployment,
  api_key=subscription_key,
  azure_endpoint=endpoint,
  api_version=api_version,
  deployment_name=deployment
)

### Testing with a correct answer

In [19]:
# Replace this with the actual output from your LLM application
#actual_output = goldens['expected_output'][0]
actual_output = 'Here are the steps you can take to get help immediately:\n\n1. Call Emergency Services: Dial 112 to reach emergency services in Amsterdam. This will connect you to the police, fire department, or medical services, depending on your needs.\n\n2. Provide Your Location: When you call, make sure to provide your exact location in Amsterdam so that help can reach you quickly.\n\n3. Stay Safe: If you are in a dangerous situation, try to find a safe place to stay while waiting for help to arrive. Avoid any confrontations and stay calm.\n\n4. Reach Out to Friends or Family: If possible, contact a trusted friend or family member who can provide support and assistance until help arrives.\n\nRemember, it is important to seek help immediately if you are scared and alone. You are not alone, and there are people who can assist you.' 
# Replace this with the expected output of your RAG generator
#expected_output = goldens['expected_output'][0]
expected_output = 'I am sorry to hear that you are scared and alone. It is important to seek help immediately. You can call emergency services in Amsterdam by dialing 112, which will connect you to the police, fire department, or medical services. They will be able to assist you and ensure that you receive the help you need. Please do not hesitate to reach out for help.'
# Replace this with the actual retrieved context from your RAG pipeline
retrieval_context = ast.literal_eval(goldens['context'][0])

metric = GEval(
    name="Tone Attunement",
    criteria="Determine whether the chatbot communication style appropriately matches the emotional register of the user's query. Consider factors such as formality, empathy, and tone consistency in your evaluation.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=custom_model
)

test_case = LLMTestCase(
    input="I am scared and alone. I urgently need a doctor in Amsterdam.",
    actual_output=actual_output,
    expected_output=expected_output,
    retrieval_context=retrieval_context
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Tone Attunement [GEval] Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...

c:\Users\dari\conda\envs\hia\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Tone Attunement [GEval] (score: 0.8, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The Actual Output provides clear, actionable steps and includes some empathetic language, such as acknowledging the user's fear and emphasizing that they are not alone. The tone is supportive and practical, which aligns with the user's urgent emotional state. However, compared to the Expected Output, the Actual Output is slightly less direct in expressing empathy at the beginning and is more procedural, which may make it feel less immediately comforting. Overall, it mostly matches the user's emotional needs but could improve by leading with a stronger empathetic statement., error: None)

For test case:

  - input: I am scared and alone. I urgently need a doctor in Amsterdam.
  - actual output: Here are the steps you can take to get help immediately:

1. Call Emergency Services: Dial 112 to reach emergency services in Amsterdam. This will conne

⚠ WARNING: No hyperparameters logged.
» ]8;id=595044;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.96s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Tone Attunement [GEval]', threshold=0.5, success=True, score=0.8, reason="The Actual Output provides clear, actionable steps and includes some empathetic language, such as acknowledging the user's fear and emphasizing that they are not alone. The tone is supportive and practical, which aligns with the user's urgent emotional state. However, compared to the Expected Output, the Actual Output is slightly less direct in expressing empathy at the beginning and is more procedural, which may make it feel less immediately comforting. Overall, it mostly matches the user's emotional needs but could improve by leading with a stronger empathetic statement.", strict_mode=False, evaluation_model='gpt-4.1-students (Azure)', error=None, evaluation_cost=None, verbose_logs='Criteria:\nDetermine whether the chatbot communication style appropriately matches the emotional register of the user\'s quer

### Testing with a wrong answer

In [20]:
# Replace this with the actual output from your LLM application
#actual_output = goldens['expected_output'][0]
actual_output = 'You can call emergency services in Amsterdam by dialing 112, which will connect you to the police, fire department, or medical services. They will be able to assist you and ensure that you receive the help you need.' 
# Replace this with the expected output of your RAG generator
#expected_output = goldens['expected_output'][0]
expected_output = 'I am sorry to hear that you are scared and alone. It is important to seek help immediately. You can call emergency services in Amsterdam by dialing 112, which will connect you to the police, fire department, or medical services. They will be able to assist you and ensure that you receive the help you need. Please do not hesitate to reach out for help.'
# Replace this with the actual retrieved context from your RAG pipeline
retrieval_context = ast.literal_eval(goldens['context'][0])

metric = GEval(
    name="Tone Attunement",
    criteria="Determine whether the chatbot communication style appropriately matches the emotional register of the user's query. Consider factors such as formality, empathy, and tone consistency in your evaluation.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=custom_model
)

test_case = LLMTestCase(
    input="I am scared and alone. I urgently need a doctor in Amsterdam.",
    actual_output=actual_output,
    expected_output=expected_output,
    retrieval_context=retrieval_context
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Tone Attunement [GEval] Metric! (using gpt-4.1-students (Azure), strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Tone Attunement [GEval] (score: 0.6, threshold: 0.5, strict: False, evaluation model: gpt-4.1-students (Azure), reason: The Actual Output provides clear and accurate information about contacting emergency services, which addresses the urgency in the user's message. However, it lacks explicit empathy and emotional acknowledgment present in the Expected Output, such as expressing concern for the user's fear and loneliness. The tone is neutral and informative but does not fully match the user's emotional register or maintain the empathetic style demonstrated in the Expected Output., error: None)

For test case:

  - input: I am scared and alone. I urgently need a doctor in Amsterdam.
  - actual output: You can call emergency services in Amsterdam by dialing 112, which will connect you to the police, fire department, or medical services. They will be able to assist you and ensure that you receive the help you need.
  - expected output: I am sorry to hear that you a

⚠ WARNING: No hyperparameters logged.
» ]8;id=584025;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.54s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Tone Attunement [GEval]', threshold=0.5, success=True, score=0.6, reason="The Actual Output provides clear and accurate information about contacting emergency services, which addresses the urgency in the user's message. However, it lacks explicit empathy and emotional acknowledgment present in the Expected Output, such as expressing concern for the user's fear and loneliness. The tone is neutral and informative but does not fully match the user's emotional register or maintain the empathetic style demonstrated in the Expected Output.", strict_mode=False, evaluation_model='gpt-4.1-students (Azure)', error=None, evaluation_cost=None, verbose_logs='Criteria:\nDetermine whether the chatbot communication style appropriately matches the emotional register of the user\'s query. Consider factors such as formality, empathy, and tone consistency in your evaluation. \n \nEvaluation Steps:\n[